# CVE/CWE → Attack Family Mapping

Dataset: `CVE_CWE_2025.csv`

## 1. Import Libraries

Start by importing what we need: pandas for data handling, plus the CWE hierarchy file.

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [2]:
cwe_df = pd.read_csv("CVE_CWE_2025.csv")
cwe_df.head()

,ID,CVE-ID,CVSS-V4,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
2,3,CVE-1999-0003,NaN,NaN,10.0,HIGH,Execute commands as root via buffer overflow i...,NVD-CWE-Other
3,4,CVE-1999-0004,NaN,NaN,5.0,MEDIUM,"MIME buffer overflow in email clients, e.g. So...",NVD-CWE-Other
4,5,CVE-1999-0005,NaN,NaN,10.0,HIGH,Arbitrary command execution via IMAP buffer ov...,NVD-CWE-Other


In [3]:
cwe_df["CWE-ID"].value_counts().head(30)

CWE-ID
NVD-CWE-Other    55550
CWE-79           35698
CWE-89           13925
CWE-119          12053
CWE-20           10851
CWE-787           9163
CWE-200           8587
CWE-352           7446
CWE-125           6786
CWE-22            6737
CWE-416           5397
CWE-264           5339
CWE-862           4441
CWE-94            4044
CWE-78            3959
CWE-476           3621
CWE-287           3360
CWE-284           3203
CWE-120           2950
CWE-434           2780
CWE-399           2637
CWE-190           2459
CWE-310           2297
CWE-400           2241
CWE-77            2162
CWE-74            1971
CWE-269           1967
CWE-121           1779
CWE-502           1707
CWE-863           1701
Name: count, dtype: int64

## 2. Build CWE/CVE → Attack Family Mapping

Attack family maps individual CWE-IDs into 13 broader attack families (Injection, XSS, 
Memory Corruption, etc.) based on the MITRE CWE hierarchy and frequency 
analysis of the top CWEs in our dataset.

In [4]:
attack_family = {
    # Injection-related
    "CWE-89": "Injection",        # SQL Injection
    "CWE-78": "Injection",        # OS Command Injection
    "CWE-77": "Injection",        # Command Injection (general)
    "CWE-94": "Injection",        # Code Injection
    "CWE-74": "Injection",        # Injection (general/base)
    "CWE-20": "Injection",        # Improper Input Validation (judgment call)
    "CWE-918": "Injection",       # SSRF
    "CWE-611": "Injection",       # XXE
    "CWE-427": "Injection",       # Uncontrolled Search Path Element
    "CWE-601": "Injection",      
    
    # Cross-Site Scripting
    "CWE-79": "XSS",

    # Memory Corruption
    "CWE-119": "Memory Corruption",  # Buffer overflow (general)
    "CWE-787": "Memory Corruption",  # Out-of-bounds Write
    "CWE-125": "Memory Corruption",  # Out-of-bounds Read
    "CWE-416": "Memory Corruption",  # Use After Free
    "CWE-476": "Memory Corruption",  # NULL Pointer Dereference
    "CWE-120": "Memory Corruption",  # Buffer Copy w/o Checking Size
    "CWE-190": "Memory Corruption",  # Integer Overflow
    "CWE-121": "Memory Corruption",  # Stack-based Buffer Overflow
    "CWE-122": "Memory Corruption",  # Heap-based Buffer Overflow
    "CWE-189": "Memory Corruption",  # Numeric Errors

    # Info Disclosure
    "CWE-200": "Info Disclosure",
    "CWE-532": "Info Disclosure",    # Insertion of Sensitive Info into Log Files

    # CSRF
    "CWE-352": "CSRF",

    # Path Traversal
    "CWE-22": "Path Traversal",
    "CWE-59": "Path Traversal",      # Improper Link Resolution (symlink attacks)

    # Authentication & Access Control
    "CWE-264": "Authentication & Access Control",  # Permissions/Privileges/Access Control
    "CWE-284": "Authentication & Access Control",  # Improper Access Control
    "CWE-862": "Authentication & Access Control",  # Missing Authorization
    "CWE-863": "Authentication & Access Control",  # Incorrect Authorization
    "CWE-269": "Authentication & Access Control",  # Improper Privilege Management
    "CWE-287": "Authentication & Access Control",  # Improper Authentication
    "CWE-306": "Authentication & Access Control",  # Missing Authentication for Critical Function
    "CWE-732": "Authentication & Access Control",  # Incorrect Permission Assignment
    "CWE-798": "Authentication & Access Control",  # Hard-coded Credentials
    "CWE-276": "Authentication & Access Control",  # Incorrect Default Permissions
    "CWE-522": "Authentication & Access Control",  # Insufficiently Protected Credentials
    "CWE-639": "Authentication & Access Control",  # Authorization Bypass via User-Controlled Key
    "CWE-255": "Authentication & Access Control",  # Credentials Management Errors

    # File Handling
    "CWE-434": "File Handling",  # Unrestricted Upload of File with Dangerous Type

    # Denial of Service
    "CWE-400": "Denial of Service",  # Uncontrolled Resource Consumption
    "CWE-399": "Denial of Service",  # Resource Management Errors
    "CWE-770": "Denial of Service",  # Allocation of Resources Without Limits
    "CWE-401": "Denial of Service",  # Missing Release of Memory (memory leak)

    # Cryptographic Issues
    "CWE-310": "Cryptographic Issues",
    "CWE-295": "Cryptographic Issues",  # Improper Certificate Validation

    # Deserialization
    "CWE-502": "Deserialization",

    # Race Conditions
    "CWE-362": "Race Condition",

    # NVD's catch-all bucket
    "NVD-CWE-Other": "Other",
}

## 3. Apply the Mapping

Apply the dictionary to create a new `attack_family` column, with any 
unmapped CWE falling back to "Other".

In [5]:
cwe_df["attack_family"] = cwe_df["CWE-ID"].map(attack_family).fillna("Other")
cwe_df["attack_family"].value_counts()

attack_family
Other                              88620
Memory Corruption                  46781
Injection                          41421
XSS                                35698
Authentication & Access Control    27247
Info Disclosure                     9377
Path Traversal                      7917
CSRF                                7446
Denial of Service                   6785
Cryptographic Issues                3341
File Handling                       2780
Deserialization                     1707
Race Condition                      1574
Name: count, dtype: int64

## 4. Checking some of the Labels

I realized that sometimes CWE labels may reflect the software weakness rather than the description, which can lead to inconsistencies between the description and the ID. So I was hoping the NLP would catch that and correct it.

In [8]:
cwe_df.columns

Index(['ID', 'CVE-ID', 'CVSS-V4', 'CVSS-V3', 'CVSS-V2', 'SEVERITY',
       'DESCRIPTION', 'CWE-ID', 'attack_family'],
      dtype='object')

In [9]:
cwe_df[["DESCRIPTION","SEVERITY", "attack_family"]].sample(10)

,DESCRIPTION,SEVERITY,attack_family
25038,Stack-based buffer overflow in the MicroWorld ...,HIGH,Other
131510,"Django 1.11.x before 1.11.19, 2.0.x before 2.0...",HIGH,Denial of Service
39166,Multiple PHP remote file inclusion vulnerabili...,MEDIUM,Injection
88191,Mediawiki before 1.28.1 / 1.27.2 / 1.23.16 con...,MEDIUM,Injection
160399,The boot loader in Das U-Boot before 2021.04-r...,HIGH,Other
59734,"opOpenSocialPlugin 0.8.2.1, > 0.9.9.2, 0.9.13,...",CRITICAL,Other
55582,pam_google_authenticator.c in the PAM module i...,LOW,Info Disclosure
70118,libavcodec/xface.h in FFmpeg before 2.5.2 esta...,HIGH,Memory Corruption
47989,Heap-based buffer overflow in the image-parsin...,HIGH,Memory Corruption
139800,In the version 12.1.0.1005 and below of 360 To...,HIGH,Injection


## 5. Define Features (X) and Target (y)

- **X**: the CVE description text (model input)
- **y**: the attack_family label and severity (what we're predicting)

In [11]:
X = cwe_df["DESCRIPTION"]
y = cwe_df[["attack_family","SEVERITY"]]

## 6. Train/Test Split

Split the data 80/20, stratified by `attack_family` to preserve class 
proportions across both sets since we have a class imbalance.

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 7. TF-IDF Vectorization

Convert text into numeric features using TF-IDF with unigrams and bigrams 
(`ngram_range=(1,2)`), So that the model can capture two-word phrases like 
"denial of service" rather than only words that are isolated from each other. 

In [27]:
vectorizer = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [28]:
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
y_train["SEVERITY"] = encoder.fit_transform(y_train[["SEVERITY"]])
y_test["SEVERITY"] = encoder.transform(y_test[["SEVERITY"]])

In [31]:
y_train_severity = y_train[["SEVERITY"]].fillna(-1)
y_test_severity = y_test[["SEVERITY"]].fillna(-1)
y_train_attack = y_train[["attack_family"]]
y_test_attack = y_test[["attack_family"]]

## 8. Train Logistic Regression Model

In [32]:
model_attack = LogisticRegression(max_iter=1000)
model_attack.fit(X_train_tfidf, y_train_attack)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(max_iter=1000)

In [33]:
model_severity = LogisticRegression(max_iter=1000)
model_severity.fit(X_train_tfidf, y_train_severity)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(max_iter=1000)

## 9. Evaluate on Test Set

In [34]:
y_pred_attack = model_attack.predict(X_test_tfidf)
print(classification_report(y_test_attack, y_pred_attack))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.68      0.61      0.64      5467
                           CSRF       0.93      0.94      0.94      1505
           Cryptographic Issues       0.87      0.67      0.76       705
              Denial of Service       0.66      0.39      0.49      1380
                Deserialization       0.91      0.70      0.79       340
                  File Handling       0.79      0.76      0.77       550
                Info Disclosure       0.67      0.49      0.57      1931
                      Injection       0.79      0.75      0.77      8299
              Memory Corruption       0.86      0.88      0.87      9330
                          Other       0.67      0.75      0.71     17619
                 Path Traversal       0.81      0.74      0.77      1606
                 Race Condition       0.77      0.58      0.66       310
                            XSS       0.91      0.

In [35]:
y_pred_severity = model_severity.predict(X_test_tfidf)
print(classification_report(y_test_severity,y_pred_severity))

              precision    recall  f1-score   support

        -1.0       0.52      0.33      0.41       443
         0.0       0.64      0.47      0.54      5629
         1.0       0.71      0.74      0.72     22376
         2.0       0.69      0.28      0.39      1970
         3.0       0.76      0.81      0.78     25721

    accuracy                           0.73     56139
   macro avg       0.66      0.53      0.57     56139
weighted avg       0.72      0.73      0.72     56139



# 10. Experimenting: Excluding "other" row

In [36]:
mapped_only = cwe_df[cwe_df["attack_family"] != "Other"]
X_mapped = mapped_only["DESCRIPTION"]
y_mapped = mapped_only["attack_family"]

In [37]:
print(f"Rows before: {len(cwe_df)}")
print(f"Rows after excluding Other: {len(mapped_only)}")

Rows before: 280694
Rows after excluding Other: 192074


In [38]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mapped, y_mapped, test_size=0.2, random_state=42, stratify=y_mapped
)

In [39]:
vectorizer_m = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1,2))
X_train_tfidf_m = vectorizer_m.fit_transform(X_train_m)
X_test_tfidf_m = vectorizer_m.transform(X_test_m)

In [40]:
model_m = LogisticRegression(max_iter=1000)

In [19]:
model_m.fit(X_train_tfidf_m, y_train_m)

LogisticRegression(max_iter=1000)

In [20]:
y_pred_m = model_m.predict(X_test_tfidf_m)

In [21]:
print(classification_report(y_test_m, y_pred_m))

                                 precision    recall  f1-score   support

Authentication & Access Control       0.79      0.87      0.83      5450
                           CSRF       0.97      0.94      0.96      1489
           Cryptographic Issues       0.88      0.72      0.79       668
              Denial of Service       0.75      0.62      0.68      1357
                Deserialization       0.94      0.73      0.82       341
                  File Handling       0.85      0.80      0.82       556
                Info Disclosure       0.77      0.70      0.73      1876
                      Injection       0.86      0.86      0.86      8284
              Memory Corruption       0.92      0.95      0.93      9356
                 Path Traversal       0.92      0.85      0.88      1583
                 Race Condition       0.90      0.68      0.77       315
                            XSS       0.98      0.97      0.98      7140

                       accuracy                  

## Summary

I tested a few variations before settling on this model on step 9:
- Baseline (unigrams, no class weighting): 0.74 accuracy
- Balanced class weights: 0.68 accuracy (higher recall on rare classes, 
  but much lower precision)
- Balanced + bigrams: 0.71 accuracy
- Bigrams only (final, above): 0.77 accuracy
- 12-class (Other excluded, step 11): 0.89 accuracy, 0.84 macro F1

In [25]:
import joblib
joblib.dump(model_m,'/Users/rehantaneja/Documents/AI4ALL/IoT-model-deployment-pipeline/NLP.joblib')

['/Users/rehantaneja/Documents/AI4ALL/IoT-model-deployment-pipeline/NLP.joblib']